# 01 - Data Cleaning and Merging
The purpose of this notebook is to load, clean, and, merge the raw workout and bodyweight datasets. The primary steps include standardizing data types, handling missing values, and combining the two datasets on the "Date" column. The final output is a clean, master DataFrame that will be used for subsequent in-depth analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', 150)  # display up to 150 rows
pd.set_option('display.max_colwidth', None)  # increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # format floats to 2 decimal places 

In [ ]:
# loading csv files
workouts_df = pd.read_csv('../data/raw/workout_data.csv')
weight_df = pd.read_csv('../data/raw/measurement_data.csv')

### 1.1 - Previewing the Workout Data
To get a better idea of the way this data is organized, let's view the head along with any attributes with null values. This will help me decide what data I can safely delete.

In [ ]:
workouts_df.head()  # preview the data

In [ ]:
workouts_df.info()

In [ ]:
workouts_df.isnull().sum()  # check how many null values are present for each column

### 1.2 - Removing Workout Metadata
We can immediately see some metadata that will not be necessary in our data analysis, so we can remove them. This includes the description and exercise notes. The title of the workout itself is also not strictly necessary in the data analysis, but may make it easier to organize specific exercises done on each day into their respective muscle groups.

In [ ]:
workouts_df = workouts_df.drop(['description', 'exercise_notes'], axis=1)  # remove the listed columns

### 1.3 - Verifying Irrelevant Exercise Data
Since we aim to look at strength training data, we can get rid of any data related to cardio or other exercise types. These exercises will not have weight or rep information. However, some exercises, like pull ups, may not have weight information since it is based on bodyweight. These exercises are vital for checking trends in strength so they should not be deleted. Thus, we will analyze the data closer to verify what data should and should not be deleted.

In [ ]:
workouts_df[workouts_df['weight_lbs'].isnull()]  # select the rows without any value for weight_lbs, but without permanently deleting the rest from the df

In [ ]:
workouts_df[workouts_df['reps'].isnull()]

### 1.4 - Deleting Irrelevant Exercise Data
Since I have verified that bodyweight exercises do not have weight information, I will remove exercises based on other attributes, like distance and duration. Then, we can check back and make sure that there are no remaining irrelevant exercises in the dataset and look at the full datasets data types using info().

In [ ]:
workouts_df[workouts_df['distance_miles'].notna()]  # select only the rows with a value for distance_miles, without permanently deleting the rest

In [ ]:
workouts_df[workouts_df['duration_seconds'].notna()]

In [ ]:
# update the df to only keep rows where distance_miles and duration_seconds are null
workouts_df = workouts_df[workouts_df['distance_miles'].isnull()]
workouts_df = workouts_df[workouts_df['duration_seconds'].isnull()]

In [ ]:
workouts_df[workouts_df['reps'].isnull()]

In [ ]:
workouts_df.info()

### 1.5 - Final Removal of Irrelevant Workout Information
Since all our cardio workouts have been removed from the dataset, we can also remove the columns that track distance and duration. Additionally, since it is not particularly pertinent to this analysis we can also get rid of superset information.

In [ ]:
workouts_df = workouts_df.drop(['superset_id', 'distance_miles', 'duration_seconds'], axis=1)

In [ ]:
workouts_df.info()

### 1.6 - Converting Workout Date Information to Datetime
To ensure that merging the datasets is a smooth process, we need to ensure that both date columns are datetime objects. However, there is a start time and an end time, something that will not be present on bodyweight data. We can keep and convert both to datetime and then later determine the workout duration if necessary for our analysis. But to ensure a common column for both datasets, we can create a new column called date based on the start_time, then convert that to datetime as well. We will also bring the date attribute to the front of the dataset, right after title, for better readability. Then we can use info() to ensure the correct data types are being used.

In [ ]:
workouts_df['start_time'] = pd.to_datetime(workouts_df['start_time'])
workouts_df['end_time'] = pd.to_datetime(workouts_df['end_time'])

In [ ]:
workouts_df['date'] = workouts_df['start_time'].dt.date
workouts_df['date'] = pd.to_datetime(workouts_df['date'])

In [ ]:
cols = workouts_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('date')))
workouts_df = workouts_df[cols]

In [ ]:
workouts_df.info()

### 1.7 - Previewing the Bodyweight Data
Now we will do much of the same data processing on the bodyweight dataset that we just did to the workout dataset. Then, after cleaning this dataset, we will be ready to merge the two datasets into the master DataFrame to be used for analysis.

In [ ]:
weight_df.head()

In [ ]:
weight_df.info()

In [ ]:
weight_df.isnull().sum()

### 1.8 - Removing Empty Measurement Columns
As we can see, most of the columns are fully empty since I only tracked my bodyweight and the date. Therefore, we can remove all the other columns, leaving only what we need.

In [ ]:
weight_df = weight_df[['date', 'weight_lbs']]

In [ ]:
weight_df.info()

### 1.9 - Converting Bodyweight Date Information to Datetime
Just like we did for the workout data, we convert the dates of when I weighed myself to be in datetime format so that it is consistent when we merge the datasets. Then, we verify the operation was successful using info().

In [ ]:
weight_df['date'] = pd.to_datetime(weight_df['date'])

In [ ]:
weight_df.info()

### 1.10 - Merging Datasets
Since both datasets are adequately cleaned and standardized, now we can merge them using the common date attribute and verify that all the data looks as expected. We will also explicitly sort the new dataset to ensure that the order is ascending to view change over time, starting from the beginning.

In [ ]:
final_df = pd.merge(workouts_df, weight_df, on='date', how='outer')  # keep all data from workouts_df and weight_df but combine the date attribute

In [ ]:
final_df.sort_values(by='date', ascending=True, inplace=True)  # sort in ascending order

In [ ]:
final_df.head()

In [ ]:
final_df.tail()

In [ ]:
final_df.info()

In [ ]:
final_df[final_df['reps'].notna()]  # checking for workout data since head and tail are all bodyweight data

### 1.11 - Fixing Issues
As we can see, there is one key issue with the data. We had two columns called weight_lbs which led to the merge renaming these to weight_lbs_x and weight_lbs_y, respectively. We will now rename these for clarity and rename all other potentially unclear column names. Additionally, since the date is what links our datasets, we will move this attribute to the front, and the weight_lbs_y column, which is our bodyweight to be right after it. This will make the data be presented more logically.

There are also two outlier data points in the very beginning of our bodyweight data. I started consistently weighing myself in April 2023, but I have a recorded weight from a single day in 2020 and another day in 2022. Since these will not be helpful in viewing any trends in bodyweight, we can remove them before saving our finalized DataFrame.

In [ ]:
final_df = final_df.rename(columns={
    'title': 'workout_title',
    'weight_lbs_x': 'working_weight_lbs',
    'weight_lbs_y': 'bodyweight_lbs'
})

In [ ]:
cols = final_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('date')))
final_df = final_df[cols]
cols.insert(1, cols.pop(cols.index('bodyweight_lbs')))
final_df = final_df[cols]

In [ ]:
final_df.head()

In [ ]:
final_df = final_df.iloc[2:].reset_index(drop=True)
final_df.head()

### 1.12 Saving the Final DataFrame
Now that we have our merged datasets and any issues have been resolved, we are ready to save it to a new file so that we can explore and analyze our master dataset in our next notebook.

In [ ]:
final_df.to_csv('../data/processed/processed_data.csv', index=False)  # index=False to prevent saving df index as a separate column

### 1.13 - Conclusion and Next Steps
In this notebook, the raw workout and bodyweight datasets were loaded, cleaned, and processed. Key steps included standardizing the 'date' column for merging, handling missing values, and removing data irrelevant to the analysis. Then, we merged the two datasets into a single, unified DataFrame and sorted chronologically. This master dataset has been saved to `data/processed/cleaned_fitness_data.csv`. Now, we will explore and analyze the data in our next notebook.